## Minimal working code

```
Our Goal:
We are not training a model or solving a real problem. Instead, we are creating a Transformer model in PyTorch and running a dummy forward pass to understand:

How to create a Transformer

What input shapes it expects

What output shapes it produces

The basic structure of the model

Think of this as "test driving" a Transformer without actually using real data. We're just making sure the engine starts and we understand the controls!

### Imports

In [ ]:
import torch
import torch.nn as nn       # includes transformer


### Creating the Transformer Model

In [ ]:
model = nn.Transformer(
    d_model=512,            # embedding size (features per word)
    nhead=8,                # number of attention heads
    num_encoder_layers=2,   # encoder layers, each layer has a multi-head attention mechanism and a feedforward neural network   
    num_decoder_layers=2    # decoder layers
)

d:\Python\.venv\Lib\site-packages\torch\nn\modules\transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


```
What is nn.Transformer?
This is PyTorch's built-in implementation of the "Attention Is All You Need" Transformer architecture. It creates a complete Transformer with both encoder and decoder.

Parameter Explanation:
Parameter	       Value	What It Means	                             Analogy
d_model	           512	    Size of word embeddings (features per word)	 Each word is represented by 512 numbers
nhead	            8	    Number of attention heads	                 8 different "perspectives" looking at relationships
num_encoder_layers	2	    How many encoder blocks stacked	             2 levels of understanding input
num_decoder_layers	2	    How many decoder blocks stacked	             2 levels of generating output

What This Creates Internally:

Transformer Model Structure:
┌─────────────────────────────────────────┐
│              TRANSFORMER                │
├─────────────────────────────────────────┤
│                                          │
│  ENCODER (2 layers):                     │
│  ┌─────────────────────────────────────┐ │
│  │ Layer 1:                            │ │
│  │   • Multi-Head Attention (8 heads)  │ │
│  │   • Feed Forward Network            │ │
│  └─────────────────────────────────────┘ │
│  ┌─────────────────────────────────────┐ │
│  │ Layer 2:                            │ │
│  │   • Multi-Head Attention (8 heads)  │ │
│  │   • Feed Forward Network            │ │
│  └─────────────────────────────────────┘ │
│                                          │
│  DECODER (2 layers):                     │
│  ┌─────────────────────────────────────┐ │
│  │ Layer 1:                            │ │
│  │   • Masked Multi-Head Attention     │ │
│  │   • Cross-Attention (with encoder)  │ │
│  │   • Feed Forward Network            │ │
│  └─────────────────────────────────────┘ │
│  ┌─────────────────────────────────────┐ │
│  │ Layer 2:                            │ │
│  │   • Masked Multi-Head Attention     │ │
│  │   • Cross-Attention (with encoder)  │ │
│  │   • Feed Forward Network            │ │
│  └─────────────────────────────────────┘ │
│                                          │
└─────────────────────────────────────────┘

Parameter Count Estimation:
Each component adds parameters. For d_model=512, nhead=8:

Encoder Layer 1:
  - Multi-Head Attention: ~512² × 3 × 8 ≈ 6.3M
  - Feed Forward: 512×2048×2 ≈ 2.1M
Encoder Layer 2: Same as Layer 1
Decoder Layer 1: Similar to encoder + cross-attention
Decoder Layer 2: Same as Layer 1

Total: ~20-30 million parameters!



### COMPLETE EXPLANATION: Attention Heads and Transformer Layers
```
PART 1: What is an Attention Head?
The Core Concept:
An attention head is one independent attention mechanism that learns to focus on different types of relationships between words. Multiple heads work in parallel, each capturing different aspects of the relationships.

Simple Analogy: Different Perspectives
Imagine you're trying to understand a complex scene:

Scene: "A tall man wearing a red hat is walking his small dog in the park"

Different people might notice different things:
┌─────────────────────────────────────────────────────────┐
│ Person 1: "Who is doing what?" → man walking dog        │
│ Person 2: "What are the attributes?" → tall man, small dog 
│ Person 3: "What are they wearing?" → red hat            │
│ Person 4: "Where are they?" → park                      │
│ Person 5: "What's the relationship?" → man with dog     │
│ Person 6: "What's the size relationship?" → tall vs small 
└─────────────────────────────────────────────────────────┘

Each person = ONE ATTENTION HEAD = ONE QKV MECHANISM
All looking at the same scene but focusing on different things!


PART 2: Visualizing Multiple Heads
Single Head View:

One attention head might focus on subject-verb relationships:
┌─────────────────────────────────────────────────────┐
│  The  │  cat  │  sat  │   on  │  the  │  mat  │
│   0.1 │  0.6  │  0.1  │  0.05 │  0.05 │  0.1  │  ← weights
│        ↑              ↑                             │
│      "cat" ←─────── "sat"                           │
└─────────────────────────────────────────────────────┘


Multi-Head View (8 heads):

Head 1 (Subject-Verb):     Head 2 (Location):         Head 3 (Attributes):
The cat sat on the mat     The cat sat on the mat     The cat sat on the mat
  0.1 0.6 0.1 0.05 0.1      0.1 0.1 0.2 0.4 0.2       0.5 0.3 0.1 0.05 0.05
  ↑                         ↑                         ↑
Focus: "cat" ← "sat"        Focus: "on" → "mat"       Focus: "The" → "cat"

Head 4 (Prepositions):      Head 5 (Articles):         Head 6 (Position):
The cat sat on the mat     The cat sat on the mat     The cat sat on the mat
0.05 0.1 0.2 0.5 0.15      0.4 0.1 0.1 0.1 0.3       0.1 0.1 0.6 0.1 0.1
  ↑                         ↑                         ↑
Focus: "sat" ← "on"         Focus: "The" ← "the"      Focus: "sat" (itself)

Head 7 (Object):            Head 8 (Sentence Flow):
The cat sat on the mat     The cat sat on the mat
0.05 0.1 0.2 0.2 0.45      0.2 0.2 0.2 0.2 0.2
  ↑                         ↑
Focus: "on" → "mat"         Focus: evenly distributed



PART 3: How Multiple Heads Work Together
Mathematical Representation:
For 8 heads, we actually have 8 separate attention calculations:


Single Head:
Attention₁(Q,K,V) = softmax(Q·Kᵀ/√d) · V  → shape: (seq_len, d_model/8)

Multi-Head (8 heads):
Head₁: Attention₁(Q₁,K₁,V₁) → captures syntax
Head₂: Attention₂(Q₂,K₂,V₂) → captures semantics
Head₃: Attention₃(Q₃,K₃,V₃) → captures positions
... up to Head₈

Final = Concatenate([Head₁, Head₂, ..., Head₈]) · Wₒ

Visual of Parallel Processing:

                    ┌─────────────────────────────────────┐
Input Word: "bank"  │                                     │
         ↓          │  Head 1: "financial meaning"        │
┌────────────────┐  │  └─► looks at "money", "account"    │
│   Project to   │  │                                     │
│ 8 Q,K,V sets   │──┼─► Head 2: "river meaning"           │
└────────────────┘  │  └─► looks at "river", "water"      │
         │          │                                     │
         │          │  Head 3: "grammatical role"         │
         └─────────┼─► └─► looks at verbs, nouns          │
                    │                                     │
                    │  ... (5 more heads)                 │
                    │                                     │
                    └─────────────────────────────────────┘
                                    ↓
                          Combined Understanding
                          "bank" = financial institution
                                   in this context


PART 4: What Each Head Learns (Research Findings)
Studies have shown that different heads specialize in different linguistic features:

Head Type	          What It Captures	           Example
Syntax Heads	      Subject-verb relationships	 "cat" ←→ "sat"
Semantic Heads	    Meaning relationships	       "bank" ←→ "money"
Position Heads	    Distance-based attention	    Nearby words matter more
Anaphora Heads	    Pronouns to nouns	           "it" ←→ "cat"
Preposition Heads	  Relationship markers	       "on" ←→ "mat"
Entity Heads	      Named entities	              "Paris" ←→ "France"
Negation Heads	    Negative markers	           "not" ←→ "good"
Coreference Heads	  Same entity mentions	       "John" ←→ "he"

PART 5: Why Multiple Heads (nhead=8)?
The Benefits:
Aspect	        Single Head	                     Multiple Heads (8)
Perspective	    One view of relationships	     8 different views
Information	    Limited to one pattern	         Captures multiple patterns
Robustness	    Can miss important connections	 Redundant, more reliable
Expressiveness	Linear combinations only	     Richer representations
Parallelization	One computation	                 8 parallel computations



PART 6: Encoder Layers (Two layer in this case) - Deep Dive

What Each Encoder Layer Does:

Input: Word embeddings with positional information(sin, cos)
                    ↓
┌─────────────────────────────────────────────────────┐
│              ENCODER LAYER 1                        │
├─────────────────────────────────────────────────────┤
│                                                     │
│  Step 1: Multi-Head Self-Attention(8 QKV mechanisms)│
│  ┌─────────────────────────────────────────────────┐│
│  │ Each word looks at ALL words in input sequence  ││
│  │ through 8 different perspectives                ││
│  │                                                 ││
│  │ "cat" ←→ "sat" (syntax)                         ││
│  │ "cat" ←→ "mat" (location?) not yet!             ││
│  │ This layer catches basic relationships          ││
│  └─────────────────────────────────────────────────┘│
│                    ↓                                │
│  Step 2: Add & Normalize (Residual connection)      │
│  └─────────────────────────────────────────────────┘│
│                    ↓                                │
│  Step 3: Feed-Forward Network                       │
│  ┌─────────────────────────────────────────────────┐│
│  │ Process each position independently             ││
│  │ Expand dimension (512 → 2048), then compress    ││
│  │ Think: "Now that I know relationships, what     ││
│  │        does this word really mean?"             ││
│  └─────────────────────────────────────────────────┘│
│                    ↓                                │
│  Step 4: Add & Normalize                            │
│  └─────────────────────────────────────────────────┘│
│                    ↓                                │
└─────────────────────────────────────────────────────┘
                     ↓
┌─────────────────────────────────────────────────────┐
│              ENCODER LAYER 2                        │
├─────────────────────────────────────────────────────┤
│  Same structure, but now works on                   │
│  more refined representations                       │
│                                                     │
│  "cat" ←→ "mat" (now possible! because layer 1      │
│   established that "sat" connects them)             │
└─────────────────────────────────────────────────────┘

Layer-by-Layer Progression:

Layer 1 Output: Basic relationships, local patterns
    "The cat sat on the mat"
     ↓  ↓   ↓   ↓  ↓   ↓
    [T][C][S][O][T][M]  (simple connections)

Layer 2 Output: More complex, abstract relationships
    "The cat sat on the mat"
     ↓  ↓   ↓   ↓  ↓   ↓
    [T-Cat][Cat-Sat][Sat-Mat]  (grouped concepts)

Layer 3+ (if more layers): Hierarchical understanding
    "The cat sat on the mat" → "Cat on mat" (complete scene)

PART 7: Decoder Layers - Deep Dive
What Each Decoder Layer Does:
Input: Target sequence (what we're generating so far)
                    ↓
┌─────────────────────────────────────────────────────┐
│              DECODER LAYER 1                        │
├─────────────────────────────────────────────────────┤
│                                                     │
│  Step 1: Masked Multi-Head Self-Attention           │
│  ┌─────────────────────────────────────────────────┐│
│ │ "I love" → predicting next word                  ││
│ │ MASK prevents looking at future words!           ││
│ │                                                  ││
│ │ Generation so far: "J'"                          ││
│ │ Can look at: "J'"                                ││
│ │ Cannot look at: "aime" (future)                  ││
│ └─────────────────────────────────────────────────┘ │
│                    ↓                                │
│  Step 2: Add & Normalize                            │
│  └─────────────────────────────────────────────────┘│
│                    ↓                                │
│  Step 3: Cross-Attention (MAGIC HAPPENS HERE!)      │
│  ┌─────────────────────────────────────────────────┐│
│ │ Decoder looks at ENCODER output!                 ││
│ │                                                  ││
│ │ Query: Current word "J'"                         ││
│ │ Keys/Values: All encoder outputs                 ││
│ │   "I" (0.8), "love" (0.7), "AI" (0.3)            ││
│ │                                                  ││
│ │ Result: "J'" strongly attends to "I"             ││
│ │ This is how translation happens!                 ││
│ └─────────────────────────────────────────────────┘ │
│                    ↓                                │
│  Step 4: Add & Normalize                            │
│  └─────────────────────────────────────────────────┘│
│                    ↓                                │
│  Step 5: Feed-Forward Network                       │
│  └─────────────────────────────────────────────────┘│
│                    ↓                                │
└─────────────────────────────────────────────────────┘
                    ↓
┌─────────────────────────────────────────────────────┐
│              DECODER LAYER 2                        │
├─────────────────────────────────────────────────────┤
│  Same structure, refines the translation            │
│  "J'" now better connected to French grammar rules  │
└─────────────────────────────────────────────────────┘

PART 8: Complete Flow Through All Layers
Example: Translating "I love AI" to "J'aime l'IA"

ENCODER PATH:
┌─────────────────────────────────────────────────────────┐
│ Input: "I love AI" (3 words, each 512-dim)              │
├─────────────────────────────────────────────────────────┤
│                                                         │
│ Encoder Layer 1:                                        │
│   Attention: "I"↔"love" (subject-verb)                  │
│              "love"↔"AI" (verb-object)                  │
│  Output: Enhanced word vectors with basic relationships │
│                                                         │
│ Encoder Layer 2:                                        │
│   Attention: "I"↔"AI" (now possible via "love")         │
│   Output: Complete understanding of the whole sentence  │
│                                                         │
│ Final Encoder Output: [enc_I, enc_love, enc_AI]         │
│         Each contains full context of the sentence      │
└─────────────────────────────────────────────────────────┘

DECODER PATH (generating "J'aime l'IA"):
┌─────────────────────────────────────────────────────────┐
│ Start: [SOS] (start token)                              │
├─────────────────────────────────────────────────────────┤
│                                                         │
│ Decoder Layer 1:                                        │
│   Step 1 (Masked Self-Attn): [SOS] only                 │
│   Step 2 (Cross-Attn): [SOS] looks at encoder           │
│      Strong attention to "I"                            │
│   Step 3 (Feed-Forward): Predict first word "J'"        │
│                                                         │
│ Decoder Layer 2:                                        │
│   Input: [SOS, J']                                      │
│   Step 1: Look at [SOS, J'] (but not future)            │
│   Step 2: Look at encoder (focus on "love")             │
│   Step 3: Predict "aime"                                │
│                                                         │
│ Continue until [EOS]                                    │
└─────────────────────────────────────────────────────────┘
PART 9: Visual Summary of All Components

┌─────────────────────────────────────────────────────────────────────┐
│                      TRANSFORMER INTERNALS                          │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  HEADS: Each is a different "lens"                                  │
│  ┌────┐ ┌────┐ ┌────┐ ┌────┐ ┌────┐ ┌────┐ ┌────┐ ┌────┐            │
│  │ H1 │ │ H2 │ │ H3 │ │ H4 │ │ H5 │ │ H6 │ │ H7 │ │ H8 │            │
│  │Syn-│ │Se- │ │Po- │ │An- │ │Pre-│ │En- │ │Ne- │ │Co- │            │
│  │tax │ │man-│ │si- │ │aph-│ │pos-│ │ti- │ │ga- │ │ref-│            │
│  │    │ │tic │ │tion│ │ora │ │ition││ty  │ │tion│ │er- │            │
│  └────┘ └────┘ └────┘ └────┘ └────┘ └────┘ └────┘ └────┘            │
│     │      │      │      │      │      │      │      │              │
│     └──────┴──────┴──────┴──────┴──────┴──────┴──────┘              │
│                              ↓                                      │
│  ENCODER LAYER 1:              ENCODER LAYER 2:                     │
│  ┌─────────────────────┐       ┌─────────────────────┐              │
│  │ Multi-Head (8)      │       │ Multi-Head (8)      │              │
│  │   Self-Attention    │──────►│   Self-Attention    │              │
│  │   Basic patterns    │       │   Complex patterns  │              │
│  └─────────────────────┘       └─────────────────────┘              │
│           │                              │                          │
│           └──────────────┬───────────────┘                          │
│                          ↓                                          │
│  DECODER LAYER 1:              DECODER LAYER 2:                     │
│  ┌─────────────────────┐       ┌─────────────────────┐              │
│  │ Masked Multi-Head   │       │ Masked Multi-Head   │              │
│  │   Self-Attention    │       │   Self-Attention    │              │
│  └─────────────────────┘       └─────────────────────┘              │
│           │                              │                          │
│  ┌─────────────────────┐       ┌─────────────────────┐              │
│  │ Cross-Attention     │       │ Cross-Attention     │              │
│  │ (looks at encoder)  │       │ (looks at encoder)  │              │
│  └─────────────────────┘       └─────────────────────┘              │
│           │                              │                          │
│           └──────────────┬───────────────┘                          │
│                          ↓                                          │
│                    Output: "J'aime l'IA"                            │
└─────────────────────────────────────────────────────────────────────┘
PART 10: Key Takeaways
About Heads:
Each head learns a different type of relationship

8 heads = 8 different perspectives on the data

Parallel processing makes them efficient

Combined they create a rich understanding

About Encoder Layers:
Layer 1: Basic, local relationships

Layer 2: More complex, global patterns

More layers = Hierarchical understanding

Output: Rich representations of input

About Decoder Layers:
Masked attention: Can't peek at future

Cross-attention: Connects to encoder

Generates output step by step

Multiple layers refine the generation

The Magic:
Heads = Different perspectives (Attention)
Layers = Deeper understanding
Together = The power of Transformers!
Remember: When you see nhead=8, think "8 experts analyzing the sentence from 8 different angles." When you see num_encoder_layers=2, think "2 levels of deepening understanding." This combination is what makes Transformers so powerful!


### Creating Dummy Input
Assume we have a source sequence and a target sequence.Each sequence represents a sequence of words, and each word is represented by a vector of 512 numbers.Input sentences are 10 words, and output sentences are 20 words because we're translating from English to French(assume).

In [3]:
src = torch.rand(10, 32, 512)   # input sequence
tgt = torch.rand(20, 32, 512)   # target sequence

```
Understanding the Shapes:
Shape format: (sequence_length, batch_size, embedding_dim)

What These Represent:

src (source/input sequence):
┌─────────────────────────────────────┐
│ Batch 0: [word1, word2, ..., word10]│
│ Batch 1: [word1, word2, ..., word10]│
│ ... (32 batches total)              │
│ Each word = 512 numbers             │
└─────────────────────────────────────┘

tgt (target/output sequence):
┌─────────────────────────────────────┐
│ Batch 0: [word1, word2, ..., word20]│
│ Batch 1: [word1, word2, ..., word20]│
│ ... (32 batches total)              │
│ Each word = 512 numbers             │
└─────────────────────────────────────┘

Real-world Analogy:
Imagine translating from English to French:
src = English sentence: "I love artificial intelligence" (10 words)
tgt = French translation: "J'aime l'intelligence artificielle" (20 words)

We process 32 different sentences at once (batch size 32)
Each word is represented by 512 numbers (embedding)

Why Random Data?
We're using random numbers (torch.rand) because:
We don't have real data for this demo
We only care about shapes, not actual values
It's faster than loading real datasets
Proves the model works with correct dimensions

### Forward Pass

In [4]:
output = model(src, tgt)

```
What Happens Inside:
Step 1: Encoder Processes src

src (10, 32, 512) 
    ↓
Encoder Layer 1:
    • Self-attention (each word looks at all 10 words)
    • Feed-forward processing
    ↓
Encoder Layer 2:
    • More self-attention
    • More processing
    ↓
Encoder Output: (10, 32, 512)  # Same shape, but transformed
    (Now contains contextual understanding)

Step 2: Decoder Processes tgt with Encoder Output

tgt (20, 32, 512) + Encoder Output
    ↓
Decoder Layer 1:
    • Masked self-attention (can't look at future words)
    • Cross-attention (looks at encoder output)
    • Feed-forward
    ↓
Decoder Layer 2:
    • Same process again
    ↓
Output: (20, 32, 512)
The "Masked" Part:
During training, when predicting the next word, the decoder can't "cheat" by looking at future words. The mask prevents this.

### Printing Shapes

In [5]:
print("SRC shape:", src.shape)
print("TGT shape:", tgt.shape)
print("OUTPUT shape:", output.shape)

SRC shape: torch.Size([10, 32, 512])
TGT shape: torch.Size([20, 32, 512])
OUTPUT shape: torch.Size([20, 32, 512])


```
Decoder preserves sequence length
Key Insight: The output shape matches the target shape, not the source shape. This makes sense because the decoder generates output of the same length as the target sequence.

### Print Model Structure

In [ ]:
print(model)   # both encoder and decoder have 2 layers, each with 8 attention heads and an embedding size of 512 features per word

Transformer(
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
        )
        (linear1): Linear(in_features=512, out_features=2048, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=2048, out_features=512, bias=True)
        (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
    (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  )
  (decoder): TransformerDecoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerDecoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=512, o

```
Encoder processes src independently
Decoder uses both tgt and encoder output
Output matches tgt length, not src length

30 million parameters even for small config
Shows why Transformers need lots of data/compute

COMMON QUESTIONS ANSWERED
Q: Why is src length (10) different from tgt length (20)?
A: In translation, source and target sentences can have different lengths. English "I love AI" (3 words) vs French "J'aime l'IA" (4 words).

Q: Why do we need batch dimension (32)?
A: Processing multiple sentences together is faster than one at a time. 32 is a common batch size.

Q: Why 512 embedding size?
A: It's a standard size that works well - large enough to capture word meanings, small enough to be practical.

Q: What would we do with real data?
A: Replace random tensors with actual word embeddings, add a loss function, and train on real translation pairs.

Q: Is this a complete working model?
A: No - it's just the architecture. Real training needs data, loss function, optimizer, and many training steps.